# Phase 05 - Copy-paste synthesis (Method A)

HSV pseudo-masks for white-background leaves and same-class field composites.
See `docs/instructions/05-copy-paste-method-a.md`.

Cells whose first line is `#| export` are copied, in order, into the
generated `group3-starvenus.py` by `tools/export_py.py`. Keep outputs cleared before committing.

## White-image pseudo-masks (T5.1)

A **pseudo-mask** is a leaf mask made by a fixed rule instead of by hand. On the white-background images the paper is
neutral (low colour saturation) and the leaf is not, so the rule is:

1. blur lightly (Gaussian, `MASK_BLUR_KSIZE`), convert to HSV;
2. **Otsu-threshold the saturation channel** (a threshold per image: the leaf is saturated, the paper is not) and AND
   the result with `V > MASK_V_MIN`, which drops near-black pixels;
3. morphological **close** (fills the pale lesions inside the leaf, which are less saturated than healthy tissue) and
   **open** (removes specks);
4. **fill holes**: flood fill the background from the border of a 1 px padded copy; whatever the flood cannot reach is
   enclosed by the leaf;
5. keep the **largest connected component**.

All thresholds are named constants fixed by hand and tuned only on images of the **training part of split 0**
(never on val or test images), so the step does not leak. Tuning log (before: 128 of the 538 training leaves of split 0 lost more than 10% of their area compared with the final
mask; after: 17 suspected failures, 3.2%, listed in `_work/results/white_mask_first_pass.txt`):

* `MASK_V_MIN`: 40 -> 20. A third of the white leaves are dark green (the 1st percentile of V inside the leaf is 35 for
  the median image), so V > 40 cut dark leaf tips and the shadowed end of leaves out of the mask. Paper shadows are
  already excluded by the saturation test, so V needs only to drop near-black pixels.
* `MASK_CLOSE_KSIZE`: 25 -> 45 px. Pale lesions that span the leaf width left gaps that cut the leaf in two (and the
  smaller piece was then dropped); 45 px closes them and also reconnects a straw-coloured tip to its leaf.

`<data-root>/_work/masks/white/<filename>.png` holds one `uint8` 0/255 mask per white image at cache resolution;
`manifest.json` there records, per image, the source sha1 and the mask area fraction, plus the recipe hash, so
`prepare` rebuilds masks only when an image or a constant changed.

**Review.** `prepare` writes contact sheets `_work/figures/pseudo_mask_qc_<k>.png` (all 769 images, outline over the
image, at most 48 per sheet). Whoever reviews them writes the filenames of the failures (shadow included, leaf tip
lost, lesion holes, two leaves merged) into `_work/masks/white_rejects.txt`, one per line (`#` starts a comment).
Rejected leaves are never used as paste sources or as segmenter targets, and show a vermilion frame on the sheets.

In [ ]:
#| export
import math
import time

MASK_BLUR_KSIZE = 5  # Gaussian blur before the HSV conversion (pixels, odd)
MASK_V_MIN = 20  # minimum brightness (V, 0-255) of a leaf pixel; only near-black pixels are dropped
MASK_CLOSE_KSIZE = 45  # elliptical closing kernel (px at the 1280 px cache scale): fills pale lesions
MASK_OPEN_KSIZE = 7  # elliptical opening kernel: removes specks
MASK_RECIPE_VERSION = 1  # bump when the algorithm (not a constant) changes
MASK_CONSTANTS = {
    "version": MASK_RECIPE_VERSION, "blur": MASK_BLUR_KSIZE, "v_min": MASK_V_MIN,
    "close": MASK_CLOSE_KSIZE, "open": MASK_OPEN_KSIZE,
}
MASK_RECIPE = hashlib.sha1(json.dumps(MASK_CONSTANTS, sort_keys=True).encode()).hexdigest()[:12]


def fill_holes(mask: np.ndarray) -> np.ndarray:
    """``mask`` (uint8 0/255) with every background region that does not touch the border filled in
    (scipy-free: flood fill the outside of a 1 px padded copy with cv2.floodFill)."""
    padded = cv2.copyMakeBorder(mask, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0)
    flooded = padded.copy()
    cv2.floodFill(flooded, np.zeros((padded.shape[0] + 2, padded.shape[1] + 2), np.uint8), (0, 0), 255)
    holes = cv2.bitwise_not(flooded)[1:-1, 1:-1]
    return mask | holes


def largest_component(mask: np.ndarray) -> np.ndarray:
    """uint8 0/255 mask reduced to its largest 8-connected component (an empty mask stays empty)."""
    count, labels, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), connectivity=8)
    if count <= 1:
        return np.zeros_like(mask)
    keep = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return ((labels == keep) * 255).astype(np.uint8)


def white_pseudo_mask(rgb: np.ndarray) -> np.ndarray:
    """Leaf mask (uint8 0/255, same height and width) of a white-background image; see the section text."""
    hsv = cv2.cvtColor(cv2.GaussianBlur(rgb, (MASK_BLUR_KSIZE, MASK_BLUR_KSIZE), 0), cv2.COLOR_RGB2HSV)
    saturation, value = hsv[..., 1], hsv[..., 2]
    threshold, _ = cv2.threshold(saturation, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (((saturation > threshold) & (value > MASK_V_MIN)).astype(np.uint8)) * 255
    ellipse = lambda size: cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))  # noqa: E731
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, ellipse(MASK_CLOSE_KSIZE))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, ellipse(MASK_OPEN_KSIZE))
    return largest_component(fill_holes(mask))


def white_masks_dir(data_root) -> Path:
    return work_dir(data_root) / "masks" / "white"


def white_mask_path(data_root, filename: str) -> Path:
    return white_masks_dir(data_root) / f"{filename}.png"


def white_rejects_path(data_root) -> Path:
    return work_dir(data_root) / "masks" / "white_rejects.txt"


def read_mask_file(path) -> np.ndarray:
    mask = cv2.imdecode(np.frombuffer(read_file_bytes(path), np.uint8), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise DatasetError(f"{path}: cannot decode mask")
    return mask


def read_white_mask(data_root, filename: str) -> np.ndarray:
    """The stored pseudo-mask (uint8 0/255) of a white image."""
    path = white_mask_path(data_root, filename)
    if not path.is_file():
        raise DatasetError(f"{path} does not exist; run `prepare` (or `run`, which builds the masks it needs)")
    return read_mask_file(path)


def _mask_one(task: tuple) -> tuple[str, float]:
    """Worker: cached image -> mask PNG. Returns (filename, mask area as a fraction of the image)."""
    filename, source, target = task
    mask = white_pseudo_mask(read_rgb(source))
    ok, buffer = cv2.imencode(".png", mask)
    if not ok:
        raise DatasetError(f"{filename}: PNG encoding failed")
    write_bytes_atomic(Path(target), buffer.tobytes())
    return filename, float((mask > 0).mean())


@dataclass
class MaskStats:
    total: int
    built: int
    reused: int


def white_masks_manifest_path(data_root) -> Path:
    return white_masks_dir(data_root) / "manifest.json"


def load_white_mask_manifest(data_root) -> dict:
    path = white_masks_manifest_path(data_root)
    if path.is_file():
        try:
            manifest = json.loads(read_file_bytes(path).decode("utf-8"))
            if manifest.get("recipe") == MASK_RECIPE:
                return manifest
        except (OSError, ValueError):
            pass
    return {"recipe": MASK_RECIPE, "files": {}}  # damaged, or made by another recipe: rebuild everything


def build_white_masks(data_root, records: list[Record], *, workers: int | None = None, log=None) -> MaskStats:
    """Pseudo-masks of the white images among ``records`` (idempotent: an up-to-date mask is kept). A mask is up to
    date when its file exists, the manifest was written by this recipe and the source sha1 is unchanged."""
    data_root = Path(data_root)
    manifest = load_white_mask_manifest(data_root)
    files = manifest["files"]
    white = [r for r in records if r.domain == "white"]

    def current(record: Record) -> bool:
        entry = files.get(record.filename)
        return bool(entry) and entry.get("sha1") == record.sha1 and white_mask_path(data_root, record.filename).is_file()

    todo = [r for r in white if not current(r)]
    sha_of = {r.filename: r.sha1 for r in todo}
    tasks = [(r.filename, str(cache_dir(data_root) / r.filename), str(white_mask_path(data_root, r.filename))) for r in todo]
    for task in tasks:
        if not Path(task[1]).is_file():
            raise DatasetError(f"{task[1]} is missing; run `prepare` to build the cache")
    done = 0
    try:
        for filename, area in run_parallel(_mask_one, tasks, workers=workers or default_workers()):
            files[filename] = {"sha1": sha_of[filename], "area": area}
            done += 1
            if log and done % 200 == 0:
                log(f"masks: {done}/{len(tasks)}")
    finally:
        if done:
            write_json(white_masks_manifest_path(data_root), {"recipe": MASK_RECIPE, "files": dict(sorted(files.items()))})
    return MaskStats(total=len(white), built=len(todo), reused=len(white) - len(todo))


def load_white_rejects(data_root, records: list[Record] | None = None) -> set[str]:
    """Filenames in ``_work/masks/white_rejects.txt`` (one per line; blank lines and ``#`` comments are ignored).
    With ``records`` every name must be a white image of the index, otherwise DatasetError names the line."""
    path = white_rejects_path(data_root)
    if not path.is_file():
        return set()
    known = {r.filename for r in records if r.domain == "white"} if records is not None else None
    rejects: set[str] = set()
    for number, line in enumerate(read_file_bytes(path).decode("utf-8").splitlines(), start=1):
        name = line.split("#", 1)[0].strip()
        if not name:
            continue
        if known is not None and name not in known:
            raise DatasetError(f"{path}, line {number}: {name!r} is not a white-background image of the dataset")
        rejects.add(name)
    return rejects

## Mask area table and the QC contact sheets

`white_mask_area.md` reports, per class, how much of a white image the leaf covers (mask pixels / image pixels, accepted
leaves only), for the problem description in the report. `pseudo_mask_qc_<k>.png` shows every white image with the
mask outline (a 1 px yellow line over a 3 px black line, so it stays visible on dark leaves and on paper), the filename
and the area fraction. The 769 images are spread evenly over `ceil(769 / 48)` = 17 sheets (45 or 46 each) on a
grid of 8 columns x 6 rows, in filename order. Leaves listed in `white_rejects.txt` get a vermilion frame.

In [ ]:
#| export
QC_COLUMNS, QC_ROWS = 8, 6  # 48 tiles per sheet at most
QC_TILE = (264, 560)  # (width, height) of an image tile in pixels, the aspect of the cached images (0.47)
QC_LABEL_HEIGHT = 24
QC_GAP = 4


def mask_area_rows(records: list[Record], areas: dict[str, float], rejects: set[str]) -> list[list]:
    """Per class [class, accepted leaves, mean %, std %, median %, min %, max %] of the mask area, then an 'all' row."""
    rows, pooled = [], []
    for cls in (*CLASS_NAMES, None):
        names = [r.filename for r in records if r.domain == "white" and (cls is None or r.cls == cls)
                 and r.filename not in rejects and r.filename in areas]
        values = 100.0 * np.array([areas[n] for n in names]) if names else np.zeros(0)
        rows.append([cls or "all", len(names), *(float(f(values)) if len(values) else float("nan") for f in
                    (np.mean, lambda v: np.std(v, ddof=1) if len(v) > 1 else float("nan"), np.median, np.min, np.max))])
    return rows


def white_mask_area_markdown(records: list[Record], areas: dict[str, float], rejects: set[str]) -> str:
    rows = mask_area_rows(records, areas, rejects)
    lines = [
        "| Class | Leaves | Mean area (%) | Std (%) | Median (%) | Min (%) | Max (%) |",
        "|---|---:|---:|---:|---:|---:|---:|",
    ]
    for name, n, mean, std, median, low, high in rows:
        cells = " | ".join("n/a" if v != v else f"{v:.1f}" for v in (mean, std, median, low, high))
        lines.append(f"| {name if name != 'all' else '**All**'} | {n} | {cells} |")
    note = (
        f"\nFraction of a white-background image covered by the pseudo-mask of the leaf, over the accepted leaves "
        f"(white images {sum(r.domain == 'white' for r in records)}, rejected in white_rejects.txt: {len(rejects)}). "
        f"Std with ddof=1.\n"
    )
    return "\n".join(lines) + "\n" + note


def write_white_mask_area(data_root, records: list[Record], rejects: set[str]) -> Path:
    areas = {name: entry["area"] for name, entry in load_white_mask_manifest(data_root)["files"].items()}
    path = results_dir(data_root) / "white_mask_area.md"
    write_text_atomic(path, white_mask_area_markdown(records, areas, rejects))
    return path


def draw_outline(image: np.ndarray, mask: np.ndarray, *, color=(255, 255, 0), halo: int = 3, line: int = 1) -> np.ndarray:
    """A copy of ``image`` (RGB) with the outline of ``mask`` (uint8 0/255, same size): a ``halo`` px black line under a
    ``line`` px coloured line."""
    out = image.copy()
    contours, _ = cv2.findContours((mask > 0).astype(np.uint8), cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, contours, -1, (0, 0, 0), halo)
    cv2.drawContours(out, contours, -1, color, line)
    return out


def qc_tile(rgb: np.ndarray, mask: np.ndarray, label: str, right: str, rejected: bool) -> np.ndarray:
    width, height = QC_TILE
    image = cv2.resize(rgb, (width, height), interpolation=cv2.INTER_AREA)
    small = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
    tile = np.full((height + QC_LABEL_HEIGHT, width, 3), 255, np.uint8)
    tile[:height] = draw_outline(image, small)
    if rejected:
        reject = tuple(int(OKABE_ITO["vermilion"][i : i + 2], 16) for i in (1, 3, 5))
        cv2.rectangle(tile, (0, 0), (width - 1, height - 1), reject, 6)
    font, scale = cv2.FONT_HERSHEY_SIMPLEX, 0.5
    cv2.putText(tile, label, (3, height + 17), font, scale, (0, 0, 0), 1, cv2.LINE_AA)
    (text_width, _), _ = cv2.getTextSize(right, font, scale, 1)
    cv2.putText(tile, right, (width - text_width - 3, height + 17), font, scale, (0, 0, 0), 1, cv2.LINE_AA)
    return tile


def qc_sheet_groups(n: int, per_sheet: int = QC_COLUMNS * QC_ROWS) -> list[tuple[int, int]]:
    """(start, stop) of each sheet: the fewest sheets that fit ``per_sheet``, filled as evenly as possible
    (769 images -> 17 sheets: 4 of 46 and 13 of 45)."""
    sheets = max(1, math.ceil(n / per_sheet))
    base, extra = divmod(n, sheets)
    groups, start = [], 0
    for k in range(sheets):
        stop = start + base + (1 if k < extra else 0)
        groups.append((start, stop))
        start = stop
    return groups


def save_mask_qc_sheets(data_root, records: list[Record], rejects: set[str] | None = None) -> list[Path]:
    """`pseudo_mask_qc_<k>.png`, k = 01.., for every white image (see the section text). Old sheets are removed."""
    rejects = rejects if rejects is not None else load_white_rejects(data_root, records)
    white = [r for r in records if r.domain == "white"]
    manifest = load_white_mask_manifest(data_root)["files"]
    out_dir = figures_dir(data_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in out_dir.glob("pseudo_mask_qc_*.png"):
        old.unlink()
    paths = []
    tile_h = QC_TILE[1] + QC_LABEL_HEIGHT
    for number, (start, stop) in enumerate(qc_sheet_groups(len(white)), start=1):
        sheet = np.full((QC_ROWS * (tile_h + QC_GAP) + QC_GAP, QC_COLUMNS * (QC_TILE[0] + QC_GAP) + QC_GAP, 3), 255, np.uint8)
        for slot, record in enumerate(white[start:stop]):
            area = manifest[record.filename]["area"]
            tile = qc_tile(read_cached_rgb(data_root, record.filename), read_white_mask(data_root, record.filename),
                           record.filename, f"{100 * area:.1f}%", record.filename in rejects)
            row, col = divmod(slot, QC_COLUMNS)
            top, left = QC_GAP + row * (tile_h + QC_GAP), QC_GAP + col * (QC_TILE[0] + QC_GAP)
            sheet[top : top + tile_h, left : left + QC_TILE[0]] = tile
        path = out_dir / f"pseudo_mask_qc_{number:02d}.png"
        ok, buffer = cv2.imencode(".png", cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
        if not ok:
            raise DatasetError(f"{path}: PNG encoding failed")
        write_bytes_atomic(path, buffer.tobytes())
        paths.append(path)
    return paths


def prepare_pseudo_masks(data_root, records: list[Record], *, workers: int | None = None, out=print) -> dict:
    """T5.1 part of `prepare`: masks of every white image (idempotent), the area table and the QC sheets."""
    started = time.time()
    if not any(r.domain == "white" for r in records):
        out("  masks       skipped: no white images")
        return {"built": 0}
    stats = build_white_masks(data_root, records, workers=workers, log=out)
    rejects = load_white_rejects(data_root, records)
    area_path = write_white_mask_area(data_root, records, rejects)
    sheets = save_mask_qc_sheets(data_root, records, rejects)
    out(
        f"  masks       {stats.total} white pseudo-masks: built {stats.built}, up to date {stats.reused}; "
        f"{len(rejects)} rejected in white_rejects.txt; {len(sheets)} QC sheets, {area_path.name} ({time.time() - started:.0f} s)"
    )
    if not rejects and not white_rejects_path(data_root).is_file():
        out("              review the sheets and list the failures in _work/masks/white_rejects.txt (T5.1)")
    return {"built": stats.built, "reused": stats.reused, "rejects": rejects, "sheets": sheets}

## Composite generator (T5.2)

A **composite** pastes a white-background leaf (cut out with its pseudo-mask) onto a field image of the **same class**.
The label is that class. The point is to give the classifier "white-domain leaf, field-domain surroundings" samples,
while removing every cue that tells a composite from a real photo (the *shortcut risk*: a network that learns the
paste edge instead of the disease). `composite_arrays` is the pure function (arrays and a `numpy.random.Generator`
in, arrays out); `make_composite` loads the records and calls it. **All randomness comes from the passed
generator**, drawn up front in a fixed order (`draw_composite_params`), so a seed reproduces a composite exactly.

| Step | What | Why |
|---|---|---|
| 1 | **Sources** (`CompositePool`): leaf from the white **train** split, not in `white_rejects.txt`; background from the field **train** split, same class, never one of the 30 annotated images of T6.1 or their duplicate-group partners. Every sampled name is checked against the split: a val or test image raises `CompositeLeakError`. | No val/test pixel may enter training. |
| 2 | **Extraction**: crop the mask's bounding box; erode the mask by 2 px (the Otsu edge still contains paper-coloured pixels, and pasting them draws a light fringe); where the leaf was cut by the edge of the white photo (touching the border), fade the alpha over 6% of the leaf length instead of leaving a straight cut end inside the field image. | The straight cut end and the fringe were the two cues visible at first sight (first contact sheet). |
| 3 | **Geometry**: rotate +-25 degrees; scale so the leaf length (longer side of its box) is 55-95% of the background height; centre within the middle 50% of the frame; whatever leaves the frame is clipped. | Field photographers centred the affected leaf (D7). |
| 4 | **Defocus**: Gaussian blur of the background, sigma in [0, 4] px at 1280 px scale. | The salient leaf is in focus, the surroundings are not; also weakens competing lesions in the background. |
| 5 | **Colour**: in LAB shift the leaf's mean L toward the mean L of the background around it (blend factor 0.3-0.7), and apply a gain of +-5% to each RGB channel of the leaf. | White images are indoor, field images outdoor at any hour: brightness alone would reveal the paste. |
| 6 | **Feathering**: blur the alpha with sigma in [1.5, 3] px, then alpha-blend. | Removes the hard cut-out edge. |
| 7 | **Shadow** (p = 0.3): a darkened, blurred, offset copy of the alpha under the leaf. | Leaves in a canopy cast soft shadows on what is behind them. |
| 8 | Output at the background's resolution (the 1280 px cache): the composite, and the binary mask (`alpha > 0.5`, `uint8` 0/255) that the segmenter of phase 06 learns from. | |

`composites_examples.png` (report figure) shows 10 composites, 2 per class, with their source leaf and background.
`composites_realism_check.png` is the **realism check for a human reviewer**: 20 composites mixed with 20 real field
training images in a shuffled 5 x 8 grid, numbered 1-40 and otherwise unlabelled. The answer key
(`_work/results/composites_realism_key.json`) is written next to it and is deliberately not drawn in the figure: do not
open it before guessing. If the composites are trivially identifiable, iterate on steps 4-6.

In [ ]:
#| export
COMPOSITE_ROTATION_DEG = 25.0  # step 3: leaf rotation, uniform in +-this
COMPOSITE_LEAF_LENGTH = (0.55, 0.95)  # step 3: leaf length as a fraction of the background height
COMPOSITE_CENTER = (0.25, 0.75)  # step 3: the leaf centre lies in the middle 50% of the frame (both axes)
COMPOSITE_BLUR_SIGMA = (0.0, 4.0)  # step 4: background defocus (px at 1280 px scale)
COMPOSITE_L_BLEND = (0.5, 0.7)  # step 5: how far the leaf's mean L moves toward the background's (spec: 0.3-0.7, upper half used)
COMPOSITE_WB_GAIN = 0.05  # step 5: per-channel leaf gain in 1 +- this
COMPOSITE_FEATHER_SIGMA = (1.5, 3.0)  # step 6: alpha blur (px)
COMPOSITE_CHROMA_EPS = 15.0  # step 5: damping of the chroma rescaling that follows a lightness change
COMPOSITE_CHROMA_LIMITS = (0.8, 2.0)  # step 5: the chroma factor stays within these bounds
COMPOSITE_EDGE_ERODE = 2  # step 2: px; the pseudo-mask edge still holds paper-coloured pixels
COMPOSITE_CUT_FADE = 0.12  # step 2: taper length at a cut end, as a fraction of the leaf length
COMPOSITE_SHADOW_P = 0.3  # step 7
COMPOSITE_SHADOW_OFFSET = (6.0, 16.0)  # step 7: px
COMPOSITE_SHADOW_SIGMA = (4.0, 9.0)  # step 7: px
COMPOSITE_SHADOW_STRENGTH = (0.25, 0.5)  # step 7
COMPOSITE_TRAIN_SCALE = 0.5  # CompositeSource composes at this fraction of the 1280 px cache (the network sees 448x224 anyway)
COMPOSITE_RECIPE_VERSION = 1
COMPOSITE_CONSTANTS = {
    "version": COMPOSITE_RECIPE_VERSION, "masks": MASK_RECIPE, "train_scale": COMPOSITE_TRAIN_SCALE, "rotation": COMPOSITE_ROTATION_DEG,
    "length": COMPOSITE_LEAF_LENGTH, "center": COMPOSITE_CENTER, "blur": COMPOSITE_BLUR_SIGMA,
    "l_blend": COMPOSITE_L_BLEND, "wb_gain": COMPOSITE_WB_GAIN, "feather": COMPOSITE_FEATHER_SIGMA,
    "chroma": (COMPOSITE_CHROMA_EPS, COMPOSITE_CHROMA_LIMITS), "erode": COMPOSITE_EDGE_ERODE, "cut_fade": COMPOSITE_CUT_FADE, "shadow_p": COMPOSITE_SHADOW_P,
    "shadow_offset": COMPOSITE_SHADOW_OFFSET, "shadow_sigma": COMPOSITE_SHADOW_SIGMA,
    "shadow_strength": COMPOSITE_SHADOW_STRENGTH,
}
COMPOSITE_RECIPE = hashlib.sha1(json.dumps(COMPOSITE_CONSTANTS, sort_keys=True).encode()).hexdigest()[:12]


@dataclass(frozen=True)
class CompositeParams:
    """Every random choice of one composite (drawn up front, in this order, from one generator)."""

    angle: float
    length_fraction: float
    center: tuple[float, float]  # (x, y) as fractions of the background width and height
    blur_sigma: float
    l_blend: float
    gains: tuple[float, float, float]
    feather_sigma: float
    shadow: bool
    shadow_angle: float
    shadow_offset: float
    shadow_sigma: float
    shadow_strength: float


def draw_composite_params(rng: np.random.Generator) -> CompositeParams:
    u = rng.uniform
    return CompositeParams(
        angle=float(u(-COMPOSITE_ROTATION_DEG, COMPOSITE_ROTATION_DEG)),
        length_fraction=float(u(*COMPOSITE_LEAF_LENGTH)),
        center=(float(u(*COMPOSITE_CENTER)), float(u(*COMPOSITE_CENTER))),
        blur_sigma=float(u(*COMPOSITE_BLUR_SIGMA)),
        l_blend=float(u(*COMPOSITE_L_BLEND)),
        gains=tuple(float(g) for g in u(1 - COMPOSITE_WB_GAIN, 1 + COMPOSITE_WB_GAIN, size=3)),
        feather_sigma=float(u(*COMPOSITE_FEATHER_SIGMA)),
        shadow=bool(rng.random() < COMPOSITE_SHADOW_P),
        shadow_angle=float(u(0.0, 2 * np.pi)),
        shadow_offset=float(u(*COMPOSITE_SHADOW_OFFSET)),
        shadow_sigma=float(u(*COMPOSITE_SHADOW_SIGMA)),
        shadow_strength=float(u(*COMPOSITE_SHADOW_STRENGTH)),
    )


def taper_cut_ends(alpha: np.ndarray, touches: tuple[bool, bool, bool, bool], fade: float) -> np.ndarray:
    """Taper the leaf to a point over ``fade`` px at every side of the crop that touched the border of the white photo
    (top, bottom, left, right), so that a leaf the photo cut off does not end in a straight edge inside the field
    image. A pixel survives where its distance to the paper is at least ``(1 - ramp)`` times the half-width of the
    leaf in that row (column), with ``ramp`` going 1 -> 0 over ``fade`` px toward the border."""
    if fade < 1.0 or not any(touches):
        return alpha
    height, width = alpha.shape
    distance = cv2.distanceTransform((alpha > 0.5).astype(np.uint8), cv2.DIST_L2, 3)
    keep = np.ones((height, width), bool)
    top, bottom, left, right = touches
    for sides, count, along_rows in (((top, bottom), height, True), ((left, right), width, False)):
        if not any(sides):
            continue
        index = np.arange(count, dtype=np.float32)
        ramp = np.ones(count, np.float32)
        if sides[0]:
            ramp = np.minimum(ramp, (index + 0.5) / fade)
        if sides[1]:
            ramp = np.minimum(ramp, (count - index - 0.5) / fade)
        ramp = np.clip(ramp, 0.0, 1.0)
        half_width = distance.max(axis=1 if along_rows else 0)
        threshold = (1.0 - ramp) * half_width
        keep &= distance >= (threshold[:, None] if along_rows else threshold[None, :])
    return alpha * keep


def rotate_patch(rgb: np.ndarray, alpha: np.ndarray, angle: float) -> tuple[np.ndarray, np.ndarray]:
    """Rotate a leaf patch (uint8 RGB and float alpha) by ``angle`` degrees on a canvas that is large enough to hold it."""
    height, width = alpha.shape
    radians = np.deg2rad(abs(angle))
    new_w = int(np.ceil(width * np.cos(radians) + height * np.sin(radians))) + 2
    new_h = int(np.ceil(width * np.sin(radians) + height * np.cos(radians))) + 2
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    matrix[0, 2] += new_w / 2 - width / 2
    matrix[1, 2] += new_h / 2 - height / 2
    rotated_rgb = cv2.warpAffine(rgb, matrix, (new_w, new_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    rotated_alpha = cv2.warpAffine(alpha, matrix, (new_w, new_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return rotated_rgb, rotated_alpha


def composite_arrays(leaf_rgb: np.ndarray, leaf_mask: np.ndarray, bg_rgb: np.ndarray, rng: np.random.Generator,
                     params: CompositeParams | None = None, *, scale: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """Steps 2-8 of the section above. ``leaf_rgb`` / ``leaf_mask`` are a white-background image and its pseudo-mask
    (uint8 0/255), ``bg_rgb`` the field image. Returns (composite uint8 RGB, mask uint8 0/255), both of the
    background's size. ``params`` (default: drawn from ``rng``) exists so that tests can fix single choices.

    ``scale`` is the resolution of the three inputs relative to the 1280 px cache (1.0 = cache resolution, 0.5 =
    half): every constant in pixels (blur, erode, feather, shadow) is multiplied by it, so a half-resolution
    composite is the same picture at a quarter of the cost. `CompositeSource` uses it for training."""
    p = params or draw_composite_params(rng)
    p = replace(p, blur_sigma=p.blur_sigma * scale, feather_sigma=p.feather_sigma * scale,
                shadow_offset=p.shadow_offset * scale, shadow_sigma=p.shadow_sigma * scale)
    frame_h, frame_w = bg_rgb.shape[:2]
    # 2: crop the leaf, pull the alpha edge inside the leaf, taper the ends the white photo cut off
    x, y, w, h = cv2.boundingRect((leaf_mask > 0).astype(np.uint8))
    if w == 0 or h == 0:
        raise DatasetError("the leaf mask is empty")
    erode = max(1, round(COMPOSITE_EDGE_ERODE * scale)) if COMPOSITE_EDGE_ERODE else 0
    eroded = cv2.erode(leaf_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * erode + 1, 2 * erode + 1))) if erode else leaf_mask
    alpha = (eroded[y : y + h, x : x + w] > 0).astype(np.float32)
    rgb = np.ascontiguousarray(leaf_rgb[y : y + h, x : x + w])
    touches = (y == 0, y + h == leaf_mask.shape[0], x == 0, x + w == leaf_mask.shape[1])
    alpha = taper_cut_ends(alpha, touches, COMPOSITE_CUT_FADE * max(h, w))
    # 3: scale to the target length, rotate, place
    fit = p.length_fraction * frame_h / max(h, w)
    new_w, new_h = max(1, round(w * fit)), max(1, round(h * fit))
    interpolation = cv2.INTER_AREA if fit < 1.0 else cv2.INTER_LINEAR
    rgb = cv2.resize(rgb, (new_w, new_h), interpolation=interpolation)
    alpha = cv2.resize(alpha, (new_w, new_h), interpolation=interpolation)
    rgb, alpha = rotate_patch(rgb, alpha, p.angle)
    patch_h, patch_w = alpha.shape
    x0 = round(p.center[0] * frame_w - patch_w / 2)
    y0 = round(p.center[1] * frame_h - patch_h / 2)
    # 4: defocused background (the whole frame)
    bg = cv2.GaussianBlur(bg_rgb, (0, 0), p.blur_sigma) if p.blur_sigma > 0.1 else bg_rgb
    # the region that can change: the patch plus room for the feather and the shadow
    margin = int(np.ceil(3 * max(p.feather_sigma, p.shadow_sigma if p.shadow else 0) + (p.shadow_offset if p.shadow else 0))) + 1
    rx0, ry0 = max(0, x0 - margin), max(0, y0 - margin)
    rx1, ry1 = min(frame_w, x0 + patch_w + margin), min(frame_h, y0 + patch_h + margin)
    if rx1 <= rx0 or ry1 <= ry0:
        raise DatasetError("the leaf was placed outside the frame")
    roi_h, roi_w = ry1 - ry0, rx1 - rx0
    alpha_roi = np.zeros((roi_h, roi_w), np.float32)
    leaf_roi = np.zeros((roi_h, roi_w, 3), np.uint8)
    px0, py0 = max(rx0, x0), max(ry0, y0)  # overlap of the patch with the region, in frame coordinates
    px1, py1 = min(rx1, x0 + patch_w), min(ry1, y0 + patch_h)
    if px1 <= px0 or py1 <= py0:
        raise DatasetError("the leaf was placed outside the frame")
    alpha_roi[py0 - ry0 : py1 - ry0, px0 - rx0 : px1 - rx0] = alpha[py0 - y0 : py1 - y0, px0 - x0 : px1 - x0]
    leaf_roi[py0 - ry0 : py1 - ry0, px0 - rx0 : px1 - rx0] = rgb[py0 - y0 : py1 - y0, px0 - x0 : px1 - x0]
    # 6: feather
    alpha_roi = np.clip(cv2.GaussianBlur(alpha_roi, (0, 0), p.feather_sigma), 0.0, 1.0)
    bg_roi = bg[ry0:ry1, rx0:rx1].astype(np.float32) / 255.0
    # 5: harmonise the leaf's brightness with the background around it, then the white-balance gain
    leaf_f = np.clip(leaf_roi.astype(np.float32) / 255.0 * np.asarray(p.gains, np.float32), 0.0, 1.0)
    leaf_lab = cv2.cvtColor(leaf_f, cv2.COLOR_RGB2LAB)
    weight = alpha_roi > 0.5
    if weight.any():
        leaf_l = float(leaf_lab[..., 0][weight].mean())
        bg_l = float(cv2.cvtColor(bg_roi, cv2.COLOR_RGB2LAB)[..., 0].mean())
        lightness = np.clip(leaf_lab[..., 0] + p.l_blend * (bg_l - leaf_l), 0.0, 100.0)
        # chroma follows lightness (a brighter leaf keeps its saturation instead of turning grey); the hue is unchanged
        chroma_scale = np.clip((lightness + COMPOSITE_CHROMA_EPS) / (leaf_lab[..., 0] + COMPOSITE_CHROMA_EPS), *COMPOSITE_CHROMA_LIMITS)
        leaf_lab[..., 1:] *= chroma_scale[..., None]
        leaf_lab[..., 0] = lightness
        leaf_f = np.clip(cv2.cvtColor(leaf_lab, cv2.COLOR_LAB2RGB), 0.0, 1.0)
    # 7: soft shadow under the leaf
    if p.shadow:
        dx, dy = p.shadow_offset * np.cos(p.shadow_angle), p.shadow_offset * np.sin(p.shadow_angle)
        shifted = cv2.warpAffine(alpha_roi, np.float32([[1, 0, dx], [0, 1, dy]]), (roi_w, roi_h), borderValue=0)
        shadow = cv2.GaussianBlur(shifted, (0, 0), p.shadow_sigma) * p.shadow_strength
        bg_roi = bg_roi * (1.0 - shadow * (1.0 - alpha_roi))[..., None]
    # 6 (blend) and 8: composite and mask
    blended = bg_roi * (1.0 - alpha_roi[..., None]) + leaf_f * alpha_roi[..., None]
    out = bg.copy()
    out[ry0:ry1, rx0:rx1] = np.clip(blended * 255.0 + 0.5, 0, 255).astype(np.uint8)
    mask = np.zeros((frame_h, frame_w), np.uint8)
    mask[ry0:ry1, rx0:rx1] = (alpha_roi > 0.5).astype(np.uint8) * 255
    return out, mask

In [ ]:
#| export
class CompositeLeakError(DatasetError):
    """A composite source that is not allowed: a val/test image, a wrong domain or class, or an annotated image."""


def annotation_exclusion_set(records: list[Record]) -> set[str]:
    """The field images that never serve as a background: the 30 annotated images of T6.1 and their near-duplicate
    partners. Empty when the dataset has too few field images to select from (tiny test trees)."""
    try:
        selection = annotation_selection(records)
    except DatasetError:
        return set()
    return annotation_exclusions(selection, globals().get("DUPLICATE_GROUPS", []))


class CompositePool:
    """The images a split may use to build composites (step 1): leaves from the white **train** part (not rejected, with
    a background of their class available) and backgrounds from the field **train** part (same class as the leaf, not
    annotated). Every name is checked against the split, here and again at every sample."""

    def __init__(self, data_root, split_seed: int, records: list[Record] | None = None, rejects=None, exclusions=None,
                 part: str = "train"):
        self.data_root = Path(data_root)
        records = load_index(data_root) if records is None else records
        split = load_split(data_root, split_seed)
        if part not in ("train", "val"):
            raise ValueError(f"part must be 'train' or 'val', got {part!r}")
        self.split_seed, self.part = split_seed, part
        self.train = frozenset(split[part])  # the part the sources come from ("train" for training, "val" for the segmenter's checkpoint selection)
        self.held_out = frozenset(name for other in ("train", "val", "test") if other != part for name in split[other])
        self.rejects = frozenset(load_white_rejects(data_root, records) if rejects is None else rejects)
        self.exclusions = frozenset(annotation_exclusion_set(records) if exclusions is None else exclusions)
        self.white_train = [r for r in records if r.domain == "white" and r.filename in self.train]
        self.backgrounds: dict[int, list[Record]] = {}
        for r in records:
            if r.domain == "field" and r.filename in self.train and r.filename not in self.exclusions:
                self.backgrounds.setdefault(r.cls_idx, []).append(r)
        self.leaves = [r for r in self.white_train if r.filename not in self.rejects and self.backgrounds.get(r.cls_idx)]
        for r in (*self.leaves, *(b for group in self.backgrounds.values() for b in group)):
            self.check_name(r)

    def check_name(self, record: Record) -> None:
        if record.filename in self.held_out or record.filename not in self.train:
            raise CompositeLeakError(f"{record.filename} is not in the {self.part} part of split {self.split_seed}: "
                                     "a composite only uses images of that part (no validation or test image may enter training)")

    def check_pair(self, leaf: Record, background: Record) -> None:
        self.check_name(leaf)
        self.check_name(background)
        if leaf.domain != "white" or background.domain != "field":
            raise CompositeLeakError(f"a composite needs a white leaf on a field background, got {leaf.domain} on {background.domain}")
        if leaf.cls_idx != background.cls_idx:
            raise CompositeLeakError(f"{leaf.filename} ({leaf.cls}) cannot go on {background.filename} ({background.cls}): classes differ")
        if leaf.filename in self.rejects or background.filename in self.exclusions:
            raise CompositeLeakError(f"{leaf.filename} / {background.filename} is rejected or annotated")

    def sample_background(self, leaf: Record, rng: np.random.Generator) -> Record:
        group = self.backgrounds[leaf.cls_idx]
        return group[int(rng.integers(len(group)))]

    def sample(self, rng: np.random.Generator, cls_idx: int | None = None) -> tuple[Record, Record]:
        """A random (leaf, background) pair of one class (any class when ``cls_idx`` is None), checked."""
        pool = self.leaves if cls_idx is None else [r for r in self.leaves if r.cls_idx == cls_idx]
        if not pool:
            raise ExperimentError(f"no white leaves with a field background to build composites from (class {cls_idx})")
        leaf = pool[int(rng.integers(len(pool)))]
        background = self.sample_background(leaf, rng)
        self.check_pair(leaf, background)
        return leaf, background


def load_leaf(data_root, record: Record, scale: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """(white-background image, pseudo-mask uint8 0/255) at ``scale`` times the cache resolution."""
    rgb, mask = read_cached_rgb(data_root, record.filename), read_white_mask(data_root, record.filename)
    if scale != 1.0:
        rgb = cv2.resize(rgb, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        mask = ((cv2.resize(mask, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) > 127) * 255).astype(np.uint8)
    return rgb, mask


def load_background(data_root, record: Record, scale: float = 1.0) -> np.ndarray:
    rgb = read_cached_rgb(data_root, record.filename)
    return rgb if scale == 1.0 else cv2.resize(rgb, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)


def make_composite(leaf_rec: Record, bg_rec: Record, rng: np.random.Generator, *, data_root,
                   scale: float = 1.0) -> tuple[np.ndarray, np.ndarray, int]:
    """T5.2: (composite image, mask, label). ``leaf_rec`` is a white image (its pseudo-mask is read from
    ``_work/masks/white``), ``bg_rec`` a field image of the same class, which is also the label's class index.
    The image has the background's size at ``scale`` x the cache resolution; the mask is uint8 0/255
    (``alpha > 0.5``). All randomness comes from ``rng``. Use ``CompositePool.sample`` to choose the records: it
    guarantees train-only, same-class, non-annotated sources."""
    if leaf_rec.cls_idx != bg_rec.cls_idx:
        raise CompositeLeakError(f"{leaf_rec.filename} ({leaf_rec.cls}) and {bg_rec.filename} ({bg_rec.cls}) differ in class")
    leaf_rgb, leaf_mask = load_leaf(data_root, leaf_rec, scale)
    image, mask = composite_arrays(leaf_rgb, leaf_mask, load_background(data_root, bg_rec, scale), rng, scale=scale)
    return image, mask, bg_rec.cls_idx


class CompositeSource:
    """Method A's extra training samples (T5.3), plugged into `LeafDataset(extra_sources=...)`.

    ``source[i]`` for ``0 <= i < n_per_epoch`` is composite number ``i`` of an epoch; ``load(i, epoch)`` returns
    ``(uint8 RGB, class index, name)`` like the other sources, and the dataset passes it through the same training
    augmentation as a real image. Every epoch is new: the random generator of epoch ``e`` is ``default_rng(seed *
    1000 + e)``; it permutes the accepted white leaves (so each leaf is used about once per epoch when
    ``n_per_epoch`` is the white-train count, a subset when it is smaller), and sample ``i`` draws its background and
    all composite choices from ``default_rng([seed * 1000 + e, i])``. A sample is therefore a function of
    ``(seed, epoch, i)`` only, whatever worker process makes it.

    Composites are made at ``scale`` x the cache resolution (default `COMPOSITE_TRAIN_SCALE`): the network sees
    448x224, so half the cache resolution costs nothing in the picture and a quarter in compute. Decoded sources are
    kept in a per-process cache of at most ``cache_mb`` megabytes (reading three images would otherwise cost more than
    the composition itself)."""

    def __init__(self, data_root, split_seed: int, n_per_epoch: int | None = None, seed: int = 0, *,
                 scale: float = COMPOSITE_TRAIN_SCALE, pool: CompositePool | None = None, cache_mb: float = 900.0):
        self.data_root = Path(data_root)
        self.pool = pool if pool is not None else CompositePool(data_root, split_seed)
        if not self.pool.leaves:
            raise ExperimentError(f"split {split_seed} has no white training leaf with a same-class field background for composites")
        self.split_seed, self.seed, self.scale = split_seed, int(seed), scale
        self.n_per_epoch = len(self.pool.white_train) if n_per_epoch is None else int(n_per_epoch)
        if self.n_per_epoch < 1:
            raise ExperimentError("n_per_epoch must be at least 1")
        self.cache_bytes = int(cache_mb * 1e6)
        self._leaf_cache: dict = {}
        self._bg_cache: dict = {}
        self._cached = 0

    def __len__(self) -> int:
        return self.n_per_epoch

    def __getstate__(self) -> dict:
        state = dict(self.__dict__)
        state.update(_leaf_cache={}, _bg_cache={}, _cached=0)  # workers start with an empty cache
        return state

    def epoch_seed(self, epoch: int) -> int:
        return self.seed * 1000 + int(epoch)

    def pair(self, index: int, epoch: int) -> tuple[Record, Record]:
        """The (leaf, background) of composite ``index`` in ``epoch``; checked against the split."""
        order = np.random.default_rng(self.epoch_seed(epoch)).permutation(len(self.pool.leaves))
        leaf = self.pool.leaves[int(order[index % len(order)])]
        background = self.pool.sample_background(leaf, np.random.default_rng([self.epoch_seed(epoch), index, 1]))
        self.pool.check_pair(leaf, background)
        return leaf, background

    def _remember(self, cache: dict, key: str, value, nbytes: int):
        if self._cached + nbytes <= self.cache_bytes:
            cache[key] = value
            self._cached += nbytes
        return value

    def _leaf(self, record: Record):
        hit = self._leaf_cache.get(record.filename)
        if hit is None:
            rgb, mask = load_leaf(self.data_root, record, self.scale)
            hit = self._remember(self._leaf_cache, record.filename, (rgb, mask), rgb.nbytes + mask.nbytes)
        return hit

    def _background(self, record: Record):
        hit = self._bg_cache.get(record.filename)
        if hit is None:
            rgb = load_background(self.data_root, record, self.scale)
            hit = self._remember(self._bg_cache, record.filename, rgb, rgb.nbytes)
        return hit

    def load_with_mask(self, index: int, epoch: int = 0) -> tuple[np.ndarray, np.ndarray, int, str]:
        """``(uint8 RGB, mask uint8 0/255 of the pasted leaf, class index, name)``; the mask is what the segmenter learns."""
        if not 0 <= index < self.n_per_epoch:
            raise IndexError(index)
        leaf, background = self.pair(index, epoch)
        rng = np.random.default_rng([self.epoch_seed(epoch), index])
        leaf_rgb, leaf_mask = self._leaf(leaf)
        image, mask = composite_arrays(leaf_rgb, leaf_mask, self._background(background), rng, scale=self.scale)
        return image, mask, background.cls_idx, f"composite:{leaf.filename}+{background.filename}"

    def load(self, index: int, epoch: int = 0) -> tuple[np.ndarray, int, str]:
        image, _, label, name = self.load_with_mask(index, epoch)
        return image, label, name

    def __getitem__(self, index: int):
        return self.load(index)

## Composite figures (T5.2)

Both figures use the training part of split 0 and a fixed seed, and compose at the full 1280 px cache resolution.

* `composites_examples.png` - 10 composites, 2 per class (rows, `CLASS_NAMES` order), each shown as
  *white leaf | field background | composite*. Report figure (section 5).
* `composites_realism_check.png` - the **blind** realism check: 20 composites (4 per class) and 20 real field training
  images (4 per class) shuffled into a 5 x 8 grid. Only the position numbers 1-40 are drawn. The key lives in
  `_work/results/composites_realism_key.json` (position -> real or composite, and the sources); **do not open it before
  you have written down your guesses**. The composites are drawn at the same resolution and through the same resize
  as the real images, so size and sharpness are no giveaway.

In [ ]:
#| export
COMPOSITE_FIGURE_SEED = 7
EXAMPLES_PER_CLASS = 2
REALISM_PER_CLASS = 4  # composites per class in the realism check; the same number of real images
REALISM_ROWS, REALISM_COLUMNS = 5, 8


def composite_examples(pool: CompositePool, seed: int = COMPOSITE_FIGURE_SEED, per_class: int = EXAMPLES_PER_CLASS) -> list[dict]:
    """``per_class`` composites of every class: dicts with the two records, the composite and its source images."""
    rng = np.random.default_rng(seed)
    rows = []
    for cls_idx in range(NUM_CLASSES):
        used: set[str] = set()
        for _ in range(per_class):
            for _attempt in range(20):  # two different leaves per class, when the pool has them
                leaf, background = pool.sample(rng, cls_idx)
                if leaf.filename not in used:
                    break
            used.add(leaf.filename)
            image, mask, _ = make_composite(leaf, background, rng, data_root=pool.data_root)
            rows.append({"cls_idx": cls_idx, "leaf": leaf, "background": background, "image": image, "mask": mask})
    return rows


def save_composites_examples(data_root, split_seed: int = 0, path: Path | None = None, pool: CompositePool | None = None) -> Path:
    """`composites_examples.png`: rows = classes, 2 examples per row, each leaf | background | composite."""
    plt = pyplot()
    pool = pool or CompositePool(data_root, split_seed)
    path = path or figures_dir(data_root) / "composites_examples.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    examples = composite_examples(pool)
    headers = ("White leaf", "Field background", "Composite")
    with plt.rc_context({"font.size": 7}):
        fig = plt.figure(figsize=(5.8, 1.78 * NUM_CLASSES + 0.3))
        grid = fig.add_gridspec(NUM_CLASSES, 3 * EXAMPLES_PER_CLASS, left=0.055, right=0.998, top=0.965, bottom=0.014,
                                wspace=0.04, hspace=0.12)
        for n, example in enumerate(examples):
            row, pair = divmod(n, EXAMPLES_PER_CLASS)
            panels = (
                (example["leaf"].filename, read_cached_rgb(data_root, example["leaf"].filename)),
                (example["background"].filename, read_cached_rgb(data_root, example["background"].filename)),
                ("", example["image"]),
            )
            for k, (caption, rgb) in enumerate(panels):
                column = 3 * pair + k
                ax = fig.add_subplot(grid[row, column])
                ax.imshow(thumbnail(rgb, 560))
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)
                if caption:
                    ax.set_xlabel(caption, fontsize=5.5, labelpad=1.5)
                if row == 0:
                    ax.set_title(headers[k], fontsize=6.5, fontweight="bold", pad=2)
                if column == 0:
                    ax.set_ylabel(CLASS_NAMES[row], fontsize=7.5, fontweight="bold", labelpad=3)
        fig.savefig(path, dpi=FIGURE_DPI)
        plt.close(fig)
    return path


def realism_items(pool: CompositePool, seed: int = COMPOSITE_FIGURE_SEED) -> list[dict]:
    """The 40 shuffled items of the realism check: ``REALISM_PER_CLASS`` composites and as many real field training
    images (never a background used by one of the composites) per class, in a random order."""
    rng = np.random.default_rng([seed, 99])
    items: list[dict] = []
    used: set[str] = set()
    for cls_idx in range(NUM_CLASSES):
        for _ in range(REALISM_PER_CLASS):
            leaf, background = pool.sample(rng, cls_idx)
            image, _, _ = make_composite(leaf, background, rng, data_root=pool.data_root)
            used.add(background.filename)
            items.append({"kind": "composite", "cls": CLASS_NAMES[cls_idx], "image": image,
                          "leaf": leaf.filename, "background": background.filename})
    for cls_idx in range(NUM_CLASSES):
        free = [r for r in pool.backgrounds.get(cls_idx, []) if r.filename not in used]
        for k in rng.permutation(len(free))[:REALISM_PER_CLASS]:
            items.append({"kind": "real", "cls": CLASS_NAMES[cls_idx], "image": read_cached_rgb(pool.data_root, free[int(k)].filename),
                          "filename": free[int(k)].filename})
    return [items[int(i)] for i in rng.permutation(len(items))]


def realism_key(items: list[dict], split_seed: int, seed: int) -> dict:
    rows = []
    for position, item in enumerate(items, start=1):
        row, col = divmod(position - 1, REALISM_COLUMNS)
        entry = {"position": position, "row": row + 1, "column": col + 1, "kind": item["kind"], "class": item["cls"]}
        entry.update({k: item[k] for k in ("leaf", "background", "filename") if k in item})
        rows.append(entry)
    return {
        "note": "ANSWER KEY of composites_realism_check.png. Do not open it before you have written down your guesses.",
        "grid": f"{REALISM_ROWS} rows x {REALISM_COLUMNS} columns, position = (row - 1) * {REALISM_COLUMNS} + column",
        "split": split_seed, "seed": seed, "composites": sum(i["kind"] == "composite" for i in items),
        "real": sum(i["kind"] == "real" for i in items), "items": rows,
    }


def save_realism_check(data_root, split_seed: int = 0, path: Path | None = None, key_path: Path | None = None,
                       pool: CompositePool | None = None, seed: int = COMPOSITE_FIGURE_SEED) -> tuple[Path, Path]:
    """`composites_realism_check.png` (40 numbered, unlabelled images) and its answer key as JSON."""
    plt = pyplot()
    pool = pool or CompositePool(data_root, split_seed)
    path = path or figures_dir(data_root) / "composites_realism_check.png"
    key_path = key_path or results_dir(data_root) / "composites_realism_key.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    items = realism_items(pool, seed)
    with plt.rc_context({"font.size": 6}):
        fig = plt.figure(figsize=(6.4, 8.35))
        grid = fig.add_gridspec(REALISM_ROWS, REALISM_COLUMNS, left=0.004, right=0.996, top=0.997, bottom=0.003, wspace=0.03, hspace=0.02)
        for n, item in enumerate(items):
            ax = fig.add_subplot(grid[n // REALISM_COLUMNS, n % REALISM_COLUMNS])
            ax.imshow(thumbnail(item["image"], 560))
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.text(0.05, 0.985, str(n + 1), transform=ax.transAxes, ha="left", va="top", fontsize=7, fontweight="bold",
                    color="white", bbox={"boxstyle": "round,pad=0.18", "fc": "black", "ec": "none", "alpha": 0.75})
        fig.savefig(path, dpi=FIGURE_DPI)
        plt.close(fig)
    write_json(key_path, realism_key(items, split_seed, seed), indent=1)
    return path, key_path


def prepare_composites(data_root, records: list[Record], out=print) -> dict:
    """T5.2 part of `prepare`: the two composite figures from split 0. Skipped when a prerequisite is missing."""
    started = time.time()
    try:
        pool = CompositePool(data_root, SPLIT_SEEDS[0], records)
        masks = load_white_mask_manifest(data_root)["files"]
        if not pool.leaves or any(r.filename not in masks for r in pool.leaves):
            raise DatasetError("pseudo-masks are missing")
    except DatasetError as exc:
        out(f"  composites  skipped: {exc}")
        return {}
    if any(len(pool.backgrounds.get(c, [])) == 0 for c in range(NUM_CLASSES)):
        out("  composites  skipped: a class has no field training image")
        return {}
    examples = save_composites_examples(data_root, SPLIT_SEEDS[0], pool=pool)
    check, key = save_realism_check(data_root, SPLIT_SEEDS[0], pool=pool)
    out(f"  composites  {len(pool.leaves)} usable white leaves ({len(pool.white_train) - len(pool.leaves)} of {len(pool.white_train)} "
        f"train leaves rejected or without a background), {sum(len(v) for v in pool.backgrounds.values())} field backgrounds; "
        f"{examples.name}, {check.name} + key ({time.time() - started:.0f} s)")
    return {"examples": examples, "realism": check, "key": key}

## Composite source for training (T5.3) and the choice of `n_per_epoch`

`CompositeSource` (above) plugs into the `extra_sources` hook of `LeafDataset`; `make_extra_sources` in
`20_classifier.ipynb` builds it for the experiments with `use_composites` (`E3-A`, `E3-A-WF`) through
`build_composite_source`. It makes sure the pseudo-masks of the white training images exist (`prepare` normally built
them; a plain `run` builds the missing ones), checks that the cache holds every source image and fixes
`n_per_epoch` = `n_factor` x (number of white training images of the split), 1.0 by default (about 538, which roughly
triples the number of field-like samples per epoch).

**The hyperparameter.** `n_factor` is compared on **split 0, validation only**, over `COMPOSITE_N_FACTORS` =
(0.5, 1.0):

```sh
python group3-starvenus.py run --exp E3-A --sweep-composites     # split 0 at 0.5x and 1x -> _work/runs/sweep/E3-A_n0.5, E3-A_n1
python group3-starvenus.py choose-composites                     # higher val mixed macro-F1 -> choices.json composite_n_factor
python group3-starvenus.py run --exp E3-A                        # all 5 splits at the chosen factor (E3-A-WF uses it too)
```

`choose-composites` writes `composite_n_factor` (value and evidence) to `choices.json`; a tie goes to 1x, the default.
Like `choose-lr`, it copies the winning sweep run to split 0 of `_work/runs/E3-A/` when it is the same training (same
config fingerprint, same split), so `run` skips split 0. If the choice is missing, `run` uses 1x and says so in its
notes and in `config.json`.

**Fingerprints.** The factor is not a field of `ExperimentConfig` (that would change the fingerprint of every finished
run). Instead the fingerprint of a `use_composites` experiment gets an extra entry, `composites: {n_factor, recipe}`,
where `recipe` hashes every constant of the mask and composite recipes. A change of the generator therefore marks
finished E3-A runs as stale (retrained, visibly), and non-composite experiments keep their fingerprints.

In [ ]:
#| export
COMPOSITE_N_FACTORS = (0.5, 1.0)  # candidates of the split-0 comparison of n_per_epoch (x the white training count)
COMPOSITE_DEFAULT_FACTOR = 1.0


def composite_factor_tag(factor: float) -> str:
    """0.5 -> '0.5', 1.0 -> '1' (sweep folder names)."""
    return f"{factor:g}"


def composite_sweep_run_name(exp_id: str, factor: float) -> str:
    return f"{exp_id}_n{composite_factor_tag(factor)}"


def composite_fingerprint_extra(config: ExperimentConfig, factor: float | None) -> dict:
    """What a composites experiment adds to its config fingerprint; ``{}`` for every other experiment, so their
    fingerprints are exactly what they were before Method A existed."""
    if not config.use_composites:
        return {}
    return {"composites": {"n_factor": COMPOSITE_DEFAULT_FACTOR if factor is None else float(factor), "recipe": COMPOSITE_RECIPE}}


def build_composite_source(data_root, split_seed: int, seed: int, *, n_factor: float = COMPOSITE_DEFAULT_FACTOR,
                           limit: int | None = None) -> CompositeSource:
    """The `CompositeSource` of one run (see the section text). ``limit`` (--fast) caps the composites per epoch."""
    records = load_index(data_root)
    split = load_split(data_root, split_seed)
    white_train = select_records(records, split["train"], ("white",))
    build_white_masks(data_root, white_train)
    pool = CompositePool(data_root, split_seed, records)
    check_cache_complete(data_root, [*pool.leaves, *(b for group in pool.backgrounds.values() for b in group)])
    n = max(1, round(n_factor * len(white_train)))
    return CompositeSource(data_root, split_seed, min(n, limit) if limit else n, seed, pool=pool)